In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import joblib

# Load data
df = pd.read_csv("history/CNC_01.csv")

# Features (input) and target (output)
#X = df[["rpm"]]              # input
#y = df["current_A"]          # output

X = df[["rpm", "temperature_C", "vibration_mm_s"]]
y = df["current_A"]

print("Starting Training...")
# Train model
model = LinearRegression()
model.fit(X, y)

# Save model
joblib.dump(model, "model_CNC_01.pkl")

print("Model trained and saved!")

Starting Training...
Model trained and saved!


In [5]:
import os
import json
import asyncio
import random
import math
from datetime import datetime, timezone

import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import joblib

# ==============================
# CONFIG
# ==============================
MACHINE_ID = "CNC_01"

# Load trained model
MODEL_PATH = f"model_{MACHINE_ID}.pkl"
model = joblib.load(MODEL_PATH)

# Alert storage
alert_log = []

app = FastAPI()

# ==============================
# SIMULATION (simple version)
# ==============================
def generate_live_reading():
    ts = datetime.now(timezone.utc)

    rpm = 1400 + random.gauss(0, 50)
    temperature = 75 + random.gauss(0, 3)
    vibration = abs(1.5 + random.gauss(0, 0.2))

    # Normal current relation
    current = 0.008 * rpm + random.gauss(0, 1)

    # Inject occasional anomaly
    if random.random() < 0.05:
        current += random.uniform(5, 10)

    return {
        "machine_id": MACHINE_ID,
        "timestamp": ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "rpm": rpm,
        "temperature_C": temperature,
        "vibration_mm_s": vibration,
        "current_A": current,
    }

# ==============================
# ML ANOMALY DETECTION
# ==============================
def detect_anomaly(reading):
    X = [[
        reading["rpm"],
        reading["temperature_C"],
        reading["vibration_mm_s"]
    ]]

    predicted = model.predict(X)[0]
    actual = reading["current_A"]

    error = abs(actual - predicted)

    threshold = 2.0  # tune later

    is_anomaly = error > threshold

    return is_anomaly, predicted, error

# ==============================
# STREAM GENERATOR
# ==============================
async def event_stream():
    while True:
        reading = generate_live_reading()

        # ML detection
        is_anomaly, predicted, error = detect_anomaly(reading)

        reading["ml_predicted_current"] = predicted
        reading["ml_error"] = error
        reading["ml_anomaly"] = is_anomaly

        # Alert trigger
        if is_anomaly:
            alert = {
                "id": len(alert_log) + 1,
                "machine_id": MACHINE_ID,
                "reason": f"ML anomaly detected (error={error:.2f})",
                "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            }
            alert_log.append(alert)

            print(f"[ALERT] {alert}")

        yield f"data: {json.dumps(reading)}\n\n"
        await asyncio.sleep(1)

# ==============================
# API ENDPOINTS
# ==============================

@app.get("/stream")
async def stream():
    return StreamingResponse(event_stream(), media_type="text/event-stream")


@app.get("/alerts")
async def get_alerts():
    return {"alerts": alert_log}


@app.get("/")
async def root():
    return {"message": "Predictive Maintenance (Single Machine + ML)"}

# ==============================
# RUN
# ==============================
if __name__ == "__main__":
    uvicorn.run("server:app", host="0.0.0.0", port=8000, reload=True)

ModuleNotFoundError: No module named 'fastapi'